# Patent Topic Modeling with LDA

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import json

from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
import tomotopy as tp
from collections import Counter
# specify which corpus version to load (used to build the file path below)
version_prefix = "v6"

## Data Loading and Preprocessing

In [38]:


# load data for the chosen prefix
base_data = Path("../../data/claims_added")
data_path = base_data / f"{version_prefix}_processed.csv"

df = pd.read_csv(data_path)

print(f"Dataset loaded: {df.shape[0]} patents")
df["text"] = df["claims"]  # downstream code uses this name

predictions_path = Path("../../data/analysis/runs/v6__full__two_stage__ts1__predictions.jsonl")
if not predictions_path.exists():
    raise FileNotFoundError(f"predictions file not found: {predictions_path}")

keep_lens_ids = set()

with open(predictions_path, "r") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        record = json.loads(line)

        if (
            record.get("status") == "success"
            and record.get("is_accelerator_hardware_design_patent") is True
        ):
            keep_lens_ids.add(record["lens_id"])

print(f"Predicted hardware design patents: {len(keep_lens_ids)}")

# ensure lens_id is string on both sides
df["lens_id"] = df["lens_id"].astype(str)

# filter main dataframe using lens_id linkage
df = df[df["lens_id"].isin(keep_lens_ids)].copy()

print(f"Filtered dataset: {df.shape[0]} patents")
df["text"] = df["claims"]  # downstream code uses this name

# Quick tensor check
tensor_count = df["text"].str.contains("tensor", case=False, na=False).sum()
print(f"Patents containing 'tensor': {tensor_count}")

# load custom stopwords from file
stopfile = Path("custom_stopwords.txt")
if not stopfile.exists():
    raise FileNotFoundError(f"custom stopwords file not found: {stopfile}")

with open(stopfile, "r") as f:
    custom_stopwords = [line.strip() for line in f if line.strip()]

# combine English stopwords with custom stopwords
stop_words = set(ENGLISH_STOP_WORDS)
for w in custom_stopwords:
    for part in w.split():
        stop_words.add(part.lower())

# text cleaning utility
def clean_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9_\s\-]", " ", s)   # keep alnum/_/-
    s = re.sub(r"\b\d{1,2}\b", " ", s)      # remove standalone 1-2 digit numbers
    s = re.sub(r"\s+", " ", s).strip()
    return s

def tokenize_unigrams(s: str) -> list[str]:
    s = clean_text(s)
    tokens = re.findall(r"\b[a-zA-Z][a-zA-Z0-9_]{1,}\b", s)
    tokens = [t for t in tokens if t not in stop_words]
    return tokens

def build_bigram_vocab(token_lists: list[list[str]], min_count: int = 100) -> set[tuple[str, str]]:
    """
    Build a vocabulary of frequent adjacent bigrams across the corpus.
    """
    bigram_counts = Counter()

    for tokens in token_lists:
        for i in range(len(tokens) - 1):
            bigram_counts[(tokens[i], tokens[i + 1])] += 1

    return {bg for bg, count in bigram_counts.items() if count >= min_count}

def add_bigrams(tokens: list[str], bigram_vocab: set[tuple[str, str]]) -> list[str]:
    """
    Keep unigrams and also append frequent bigrams as underscore-joined tokens.
    Example:
      ['tensor', 'core', 'unit'] ->
      ['tensor', 'core', 'unit', 'tensor_core', 'core_unit']  # if both are in vocab
    """
    out = list(tokens)

    for i in range(len(tokens) - 1):
        bg = (tokens[i], tokens[i + 1])
        if bg in bigram_vocab:
            out.append(tokens[i] + "_" + tokens[i + 1])

    return out

# Step 1: cleaned text
df["text_clean"] = df["text"].map(clean_text)

# Step 2: unigram tokenization
df["tokens_unigram"] = df["text"].map(tokenize_unigrams)

# drop docs that are already empty
df = df[df["tokens_unigram"].map(len) > 0].copy()

# Step 3: build corpus-wide bigram vocab
bigram_vocab = build_bigram_vocab(df["tokens_unigram"].tolist(), min_count=15)

print(f"Frequent bigrams kept: {len(bigram_vocab)}")
print("Sample bigrams:", [f"{a}_{b}" for a, b in list(sorted(bigram_vocab))[:20]])

# Step 4: add bigrams into each document
df["tokens"] = df["tokens_unigram"].map(lambda toks: add_bigrams(toks, bigram_vocab))

# final drop of any empty docs, just in case
df = df[df["tokens"].map(len) > 0].copy()

print(f"Usable documents: {len(df)}")
print(f"Average unigram tokens per doc: {df['tokens_unigram'].map(len).mean():.1f}")
print(f"Average tokens per doc after bigrams: {df['tokens'].map(len).mean():.1f}")

# quick sanity check
for i in range(min(3, len(df))):
    print(f"\nDoc {i} sample tokens:")
    print(df.iloc[i]["tokens"][:40])

Dataset loaded: 29481 patents
Predicted hardware design patents: 5309
Filtered dataset: 5309 patents
Patents containing 'tensor': 252
Frequent bigrams kept: 24043
Sample bigrams: ['a_lut_memory', 'abbreviated_math', 'ability_graphics', 'able_accept', 'able_issue', 'absolute_difference', 'absolute_differences', 'absolute_input', 'absolute_inputs', 'abutting_edge', 'acc_acc', 'accelerated_access', 'accelerated_apd', 'accelerated_data', 'accelerated_graphics', 'acceleration_acceleration', 'acceleration_boards', 'acceleration_card', 'acceleration_circuitry', 'acceleration_circuits']
Usable documents: 5309
Average unigram tokens per doc: 432.5
Average tokens per doc after bigrams: 655.0

Doc 0 sample tokens:
['servicing', 'read', 'request', 'response', 'read', 'request', 'memory', 'block', 'requester', 'having', 'providing', 'exclusive', 'access', 'requested', 'memory', 'block', 'requester', 'requested', 'memory', 'block', 'modified', 'accessed', 'previous', 'requester', 'having', 'requeste

## LDA Model Fitting

In [39]:
n_topics = 30

mdl = tp.LDAModel(
    k=n_topics,
    alpha=0.3,   # like doc_topic_prior
    eta=0.01,    # like topic_word_prior
    min_df=10,   # similar spirit to CountVectorizer(min_df=10)
    rm_top=70,    # can raise later if generic patent terms dominate
    seed=42
)

for tokens in df["tokens"]:
    mdl.add_doc(tokens)

print(f"Documents in model: {len(mdl.docs)}")
print(f"Vocabulary size: {mdl.num_vocabs}")

Documents in model: 5309
Vocabulary size: 0


In [40]:
print("docs before train:", len(mdl.docs))
print("vocabs before train:", mdl.num_vocabs)

mdl.train(1)

print("docs after train:", len(mdl.docs))
print("vocabs after train:", mdl.num_vocabs)

docs before train: 5309
vocabs before train: 0
docs after train: 5309
vocabs after train: 11959


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)


In [41]:
print("Training LDA model...")
for i in range(0, 100, 10):
    mdl.train(10)
    print(f"Iteration: {i+10}\tLog-likelihood per word: {mdl.ll_per_word:.4f}")


Training LDA model...
Iteration: 10	Log-likelihood per word: -8.1462
Iteration: 20	Log-likelihood per word: -7.8075
Iteration: 30	Log-likelihood per word: -7.6889
Iteration: 40	Log-likelihood per word: -7.6270
Iteration: 50	Log-likelihood per word: -7.5892
Iteration: 60	Log-likelihood per word: -7.5660
Iteration: 70	Log-likelihood per word: -7.5464
Iteration: 80	Log-likelihood per word: -7.5324
Iteration: 90	Log-likelihood per word: -7.5196
Iteration: 100	Log-likelihood per word: -7.5092


In [42]:
doc_topic_dist = np.array([doc.get_topic_dist() for doc in mdl.docs])

print(f"LDA fitted with {n_topics} topics")
print("Avg max topic weight:", doc_topic_dist.max(axis=1).mean())
print(f"Final log-likelihood per word: {mdl.ll_per_word:.4f}")

LDA fitted with 30 topics
Avg max topic weight: 0.4325419
Final log-likelihood per word: -7.5092


## View topics

In [43]:
topic_summaries = {}

for topic_idx in range(mdl.k):
    top_words = [w for w, _ in mdl.get_topic_words(topic_idx, top_n=15)]
    topic_summaries[topic_idx] = top_words

    print(f"Topic {topic_idx:02d}: " + ", ".join(top_words))

Topic 00: video, cache_memory, data_cache, data_storage, tag, data_stored, content, cache_line, data_memory, lines, graphics_data, data_data, memory_cache, cache_lines, cache_cache
Topic 01: circuits, multiplexer, adder, output_data, outputs, selection, selector, inputs, arithmetic, data_input, control_circuit, data_output, select, shift, end
Topic 02: format, floating, floating_point, precision, exponent, integer, mantissa, fixed, segment, length, shift, binary, sign, fixed_point, conversion
Topic 03: nodes, acceleration, fetch, flag, pipelines, child, streams, control_circuitry, execution_circuitry, end, tree, hierarchical, decoded, decode, traversal
Topic 04: kernel, feature, convolution, map, feature_map, multiplication, convolutional, zero, sparse, matrix_multiplication, input_feature, vectors, weight_matrix, matrices, matrix_matrix
Topic 05: product, multiplier, multiply, sum, multiplication, accumulator, accumulate, accumulation, partial, reconfigurable, products, multiply_accum

In [ ]:
from code.topic_modeling.lda_pipeline.stability import compute_topic_stability

k_values = [16, 18, 20, 22, 24, 26, 28]
seeds = [0, 1, 2, 3, 4]

results_df = compute_topic_stability(
    tokenized_docs=df["tokens"].tolist(),
    k_values=k_values,
    seeds=seeds,
    iterations=100,
    vocab_min_df=10,
    min_df=10,
    rm_top=0
)

print("\n===== SUMMARY =====")
print(results_df.sort_values("k"))

Fixed comparison vocab size: 29474

===== k = 16 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parall

Omega (theta-side):      0.523
Omega (words-side):      0.584
Omega (avg):             0.553
Omega SE (repo-style):   0.0293
Avg Max Topic Cosine:    0.380
Avg Mean Topic Cosine:   0.116

===== k = 18 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parall

Omega (theta-side):      0.474
Omega (words-side):      0.510
Omega (avg):             0.492
Omega SE (repo-style):   0.0168
Avg Max Topic Cosine:    0.427
Avg Mean Topic Cosine:   0.093

===== k = 20 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parall

Omega (theta-side):      0.521
Omega (words-side):      0.591
Omega (avg):             0.556
Omega SE (repo-style):   0.0189
Avg Max Topic Cosine:    0.469
Avg Mean Topic Cosine:   0.081

===== k = 22 =====


/Users/noahfehr/anaconda3/lib/python3.12/site-packages/tomotopy/models.py:637: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  return self._train(iterations, workers, parallel, freeze_topics, callback_interval, callback)
